In [ ]:
from pathlib import Path
from IPython.display import display
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from xgboost import XGBRFClassifier

from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
    GridSearchCV,
)

from sklearn.utils.class_weight import (
    compute_sample_weight
)

from sklearn.preprocessing import (
    label_binarize
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

MATRIX_PATH = Path(
    "related_files/ml_76_matrix_sens_log1p_cleaned.tsv"
)

OUTPUT_ROOT = Path(
    "results/ml_output_sens_xgb"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

LABEL_COLUMNS = [
    "sens_A",
    "sens_B",
    "sens_C",
]

SCENARIO_NAMES = {
    "sens_A": "Scenario A",
    "sens_B": "Scenario B",
    "sens_C": "Scenario C",
}

CLASS_NAMES = {
    "sens_A": {
        0: "Non-yeast-forming",
        1: "Yeast-forming",
    },
    "sens_B": {
        0: "Non-strict yeast",
        1: "Strict yeast",
    },
    "sens_C": {
        0: "Filamentous",
        1: "Strict yeast",
        2: "Dimorphic",
    },
}

RANDOM_STATE = 2026

N_SPLITS = 5
N_REPEATS = 20

GRID_N_JOBS = 5
XGB_N_JOBS = 5
RUN_SHAP = True

BLUE = "#0072B2"
ORANGE = "#D55E00"
GREEN = "#009E73"
PURPLE = "#CC79A7"
GRAY = "#666666"

CLASS_COLORS = [
    BLUE,
    ORANGE,
    GREEN,
    PURPLE,
    "#E69F00",
]

sns.set_theme(
    context="paper",
    style="ticks",
    font_scale=1.25
)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial"],
    "svg.fonttype": "none",
    "axes.linewidth": 1.0,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def save_figure(
    fig,
    output_dir,
    filename
):
    output_dir = Path(output_dir)
    fig.savefig(output_dir / f"{filename}.svg", bbox_inches="tight", facecolor="white")

    fig.savefig(
        output_dir / f"{filename}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        output_dir / f"{filename}.pdf",
        bbox_inches="tight",
        facecolor="white"
    )

df = pd.read_csv(
    MATRIX_PATH,
    sep="\t"
)

required_columns = {
    "species_name",
    *LABEL_COLUMNS
}

missing_columns = (
    required_columns - set(df.columns)
)

if missing_columns:
    raise ValueError(
        f"Input file is missing columns: {missing_columns}"
    )

if df["species_name"].duplicated().any():
    duplicated_species = df.loc[
        df["species_name"].duplicated(),
        "species_name"
    ].tolist()

    raise ValueError(
        "Duplicate species in input: "
        + ", ".join(duplicated_species)
    )

feature_names = [
    column for column in df.columns
    if column not in [
        "species_name",
        *LABEL_COLUMNS
    ]
]

feature_df = df[feature_names].apply(
    pd.to_numeric,
    errors="coerce"
)

all_missing_features = feature_df.columns[
    feature_df.isna().all()
].tolist()

if all_missing_features:
    raise ValueError(
        "Nonnumeric feature columns: \n"
        + "\n".join(all_missing_features)
    )

feature_df = feature_df.fillna(0)

X_all = feature_df.to_numpy(
    dtype=np.float32
)

species_all = df[
    "species_name"
].astype(str).to_numpy()

print(f"Species: {len(df)}")
print(f"OG features: {len(feature_names)}")

for label_column in LABEL_COLUMNS:
    print(
        f"\n{SCENARIO_NAMES[label_column]} class counts:"
    )

    print(
        df[label_column]
        .value_counts(dropna=False)
        .sort_index()
    )


In [ ]:
PARAM_GRID = {
    "n_estimators": [300, 500],
    "max_depth": [2, 3],
    "min_child_weight": [1, 2],
    "subsample": [0.6, 0.8],
    "colsample_bynode": [0.3, 0.6],
}

def make_binary_xgbrf(
    scale_pos_weight,
    random_state
):
    return XGBRFClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        importance_type="gain",
        scale_pos_weight=scale_pos_weight,
        n_jobs=XGB_N_JOBS,
        random_state=random_state,
        verbosity=0,
    )

def make_multiclass_xgbrf(
    n_classes,
    random_state
):
    return XGBRFClassifier(
        objective="multi:softprob",
        num_class=n_classes,
        eval_metric="mlogloss",
        tree_method="hist",
        importance_type="gain",
        n_jobs=XGB_N_JOBS,
        random_state=random_state,
        verbosity=0,
    )

def calculate_mean_abs_shap(
    model,
    X_test
):
    explainer = shap.TreeExplainer(
        model
    )

    try:
        shap_values = explainer.shap_values(
            X_test,
            check_additivity=False
        )
    except TypeError:
        shap_values = explainer.shap_values(
            X_test
        )

    if isinstance(shap_values, list):
        shap_array = np.stack(
            shap_values,
            axis=-1
        )
    else:
        shap_array = np.asarray(
            shap_values
        )

    if shap_array.ndim == 2:
        return np.abs(
            shap_array
        ).mean(axis=0)

    if shap_array.ndim == 3:
        if shap_array.shape[1] == X_test.shape[1]:
            return np.abs(shap_array).mean(
                axis=(0, 2)
            )

        if shap_array.shape[2] == X_test.shape[1]:
            return np.abs(shap_array).mean(
                axis=(0, 1)
            )

    raise ValueError(
        "Unsupported SHAP array shape: "
        f"{shap_array.shape}"
    )

def get_display_names(
    label_column,
    original_classes
):
    mapping = CLASS_NAMES.get(
        label_column,
        {}
    )

    return [
        mapping.get(
            original_class,
            str(original_class)
        )
        for original_class in original_classes
    ]


In [ ]:
# Tune within each outer training fold; evaluate only on its held-out species.
def run_scenario(label_column):
    scenario_name = SCENARIO_NAMES[
        label_column
    ]

    print("\n" + "=" * 70)
    print(f"Running {scenario_name}")
    print("=" * 70)

    task_dir = (
        OUTPUT_ROOT / label_column
    )

    task_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    valid_mask = df[
        label_column
    ].notna().to_numpy()

    X = X_all[valid_mask]
    species = species_all[valid_mask]

    y_original = pd.to_numeric(
        df.loc[valid_mask, label_column],
        errors="raise"
    ).astype(int).to_numpy()

    original_classes = np.sort(
        np.unique(y_original)
    )

    class_to_encoded = {
        original_class: encoded_class
        for encoded_class, original_class
        in enumerate(original_classes)
    }

    encoded_to_original = {
        encoded_class: original_class
        for original_class, encoded_class
        in class_to_encoded.items()
    }

    y = np.array([
        class_to_encoded[value]
        for value in y_original
    ])

    n_classes = len(original_classes)
    is_binary = n_classes == 2

    if n_classes not in [2, 3]:
        raise ValueError(
            f"{scenario_name} has {n_classes} classes: "
            f"{original_classes}"
        )

    class_counts = (
        pd.Series(y)
        .value_counts()
        .sort_index()
    )

    if class_counts.min() < N_SPLITS:
        raise ValueError(
            f"{scenario_name}Smallest class has "
            f"{class_counts.min()} samples; "
            f"not enough for {N_SPLITS}-fold cross-validation."
        )

    outer_cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE
    )

    fold_results = []
    prediction_tables = []
    gain_tables = []
    shap_tables = []

    binary_roc_curves = []
    binary_pr_curves = []

    multiclass_roc_curves = {
        class_id: []
        for class_id in range(n_classes)
    }

    multiclass_pr_curves = {
        class_id: []
        for class_id in range(n_classes)
    }

    for fold_id, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y),
        start=1
    ):
        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        inner_cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=RANDOM_STATE + fold_id
        )

        if is_binary:
            negative_count = np.sum(
                y_train == 0
            )

            positive_count = np.sum(
                y_train == 1
            )

            scale_pos_weight = (
                negative_count / positive_count
            )

            estimator = make_binary_xgbrf(
                scale_pos_weight=scale_pos_weight,
                random_state=RANDOM_STATE + fold_id
            )

            scoring = "average_precision"
            fit_parameters = {}

        else:
            estimator = make_multiclass_xgbrf(
                n_classes=n_classes,
                random_state=RANDOM_STATE + fold_id
            )

            scoring = "f1_macro"

            fit_parameters = {
                "sample_weight": compute_sample_weight(
                    class_weight="balanced",
                    y=y_train
                )
            }

        search = GridSearchCV(
            estimator=estimator,
            param_grid=PARAM_GRID,
            scoring=scoring,
            cv=inner_cv,
            n_jobs=GRID_N_JOBS,
            pre_dispatch=GRID_N_JOBS,
            refit=True,
            return_train_score=False,
            error_score="raise",
        )

        search.fit(
            X_train,
            y_train,
            **fit_parameters
        )

        best_model = search.best_estimator_

        test_probabilities = (
            best_model.predict_proba(X_test)
        )

        if is_binary:
            positive_probabilities = (
                test_probabilities[:, 1]
            )

            test_predictions = (
                positive_probabilities >= 0.5
            ).astype(int)

        else:
            test_predictions = (
                best_model.predict(X_test)
            ).astype(int)

        balanced_accuracy = (
            balanced_accuracy_score(
                y_test,
                test_predictions
            )
        )

        accuracy = accuracy_score(
            y_test,
            test_predictions
        )

        f1_macro = f1_score(
            y_test,
            test_predictions,
            average="macro",
            zero_division=0
        )

        precision_macro = precision_score(
            y_test,
            test_predictions,
            average="macro",
            zero_division=0
        )

        recall_macro = recall_score(
            y_test,
            test_predictions,
            average="macro",
            zero_division=0
        )

        mcc = matthews_corrcoef(
            y_test,
            test_predictions
        )

        result_row = {
            "fold": fold_id,
            "accuracy": accuracy,
            "balanced_accuracy": balanced_accuracy,
            "f1_macro": f1_macro,
            "precision_macro": precision_macro,
            "recall_macro": recall_macro,
            "mcc": mcc,
            "best_inner_score": search.best_score_,
            "best_params": json.dumps(
                search.best_params_,
                ensure_ascii=False
            ),
        }

        if is_binary:
            roc_auc = roc_auc_score(
                y_test,
                positive_probabilities
            )

            pr_auc = average_precision_score(
                y_test,
                positive_probabilities
            )

            result_row["roc_auc"] = roc_auc
            result_row["pr_auc"] = pr_auc

            fpr, tpr, _ = roc_curve(
                y_test,
                positive_probabilities
            )

            precision_curve, recall_curve, _ = (
                precision_recall_curve(
                    y_test,
                    positive_probabilities
                )
            )

            binary_roc_curves.append(
                (fpr, tpr)
            )

            binary_pr_curves.append(
                (
                    precision_curve,
                    recall_curve
                )
            )

        else:
            y_test_binary = label_binarize(
                y_test,
                classes=np.arange(n_classes)
            )

            result_row[
                "roc_auc_ovr_macro"
            ] = roc_auc_score(
                y_test_binary,
                test_probabilities,
                average="macro",
                multi_class="ovr"
            )

            result_row[
                "pr_auc_macro"
            ] = average_precision_score(
                y_test_binary,
                test_probabilities,
                average="macro"
            )

            for class_id in range(n_classes):
                class_fpr, class_tpr, _ = roc_curve(
                    y_test_binary[:, class_id],
                    test_probabilities[:, class_id]
                )

                class_precision, class_recall, _ = (
                    precision_recall_curve(
                        y_test_binary[:, class_id],
                        test_probabilities[:, class_id]
                    )
                )

                multiclass_roc_curves[
                    class_id
                ].append(
                    (class_fpr, class_tpr)
                )

                multiclass_pr_curves[
                    class_id
                ].append(
                    (
                        class_precision,
                        class_recall
                    )
                )

        fold_results.append(
            result_row
        )

        prediction_data = {
            "species_name": species[test_idx],
            "fold": fold_id,
            "true_label_encoded": y_test,
            "predicted_label_encoded": test_predictions,
            "true_label_original": [
                encoded_to_original[value]
                for value in y_test
            ],
            "predicted_label_original": [
                encoded_to_original[value]
                for value in test_predictions
            ],
        }

        for class_id in range(n_classes):
            original_class = (
                encoded_to_original[class_id]
            )

            prediction_data[
                f"probability_class_{original_class}"
            ] = test_probabilities[:, class_id]

        prediction_tables.append(
            pd.DataFrame(prediction_data)
        )

        gain_tables.append(
            pd.Series(
                best_model.feature_importances_,
                index=feature_names,
                name=f"fold_{fold_id}"
            )
        )

        if RUN_SHAP:
            mean_abs_shap = calculate_mean_abs_shap(
                best_model,
                X_test
            )

            shap_tables.append(
                pd.Series(
                    mean_abs_shap,
                    index=feature_names,
                    name=f"fold_{fold_id}"
                )
            )

        print(
            f"{scenario_name} | Fold {fold_id:>3} | "
            f"Balanced accuracy="
            f"{balanced_accuracy:.3f} | "
            f"F1-macro={f1_macro:.3f} | "
            f"MCC={mcc:.3f}"
        )

    return {
        "label_column": label_column,
        "scenario_name": scenario_name,
        "task_dir": task_dir,
        "X": X,
        "y": y,
        "species": species,
        "n_classes": n_classes,
        "is_binary": is_binary,
        "original_classes": original_classes,
        "encoded_to_original": encoded_to_original,
        "fold_df": pd.DataFrame(fold_results),
        "predictions_df": pd.concat(
            prediction_tables,
            ignore_index=True
        ),
        "gain_df": pd.concat(
            gain_tables,
            axis=1
        ),
        "shap_df": (
            pd.concat(shap_tables, axis=1)
            if RUN_SHAP
            else None
        ),
        "binary_roc_curves": binary_roc_curves,
        "binary_pr_curves": binary_pr_curves,
        "multiclass_roc_curves": multiclass_roc_curves,
        "multiclass_pr_curves": multiclass_pr_curves,
    }


In [ ]:
def save_and_plot_scenario(result):
    label_column = result["label_column"]
    scenario_name = result["scenario_name"]
    task_dir = result["task_dir"]

    y = result["y"]
    n_classes = result["n_classes"]
    is_binary = result["is_binary"]

    original_classes = result[
        "original_classes"
    ]

    fold_df = result["fold_df"]
    predictions_df = result["predictions_df"]
    gain_df = result["gain_df"]
    shap_df = result["shap_df"]

    display_names = get_display_names(
        label_column,
        original_classes
    )

    metric_candidates = [
        "roc_auc",
        "pr_auc",
        "roc_auc_ovr_macro",
        "pr_auc_macro",
        "accuracy",
        "balanced_accuracy",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "mcc",
    ]

    metric_columns = [
        metric for metric in metric_candidates
        if metric in fold_df.columns
    ]

    performance_summary = (
        fold_df[metric_columns]
        .agg([
            "mean",
            "std",
            "median",
            "min",
            "max"
        ])
        .T
    )

    fold_df.to_csv(
        task_dir / "fold_metrics.tsv",
        sep="\t",
        index=False
    )

    performance_summary.to_csv(
        task_dir / "performance_summary.tsv",
        sep="\t"
    )

    predictions_df.to_csv(
        task_dir /
        "out_of_fold_predictions.tsv",
        sep="\t",
        index=False
    )

    print(f"\n{scenario_name} performance:")
    display(performance_summary)

    fpr_grid = np.linspace(0, 1, 300)
    recall_grid = np.linspace(0, 1, 300)

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10.4, 4.5)
    )

    if is_binary:
        roc_arrays = []

        for fpr, tpr in result[
            "binary_roc_curves"
        ]:
            interpolated = np.interp(
                fpr_grid,
                fpr,
                tpr
            )

            interpolated[0] = 0
            roc_arrays.append(interpolated)

        roc_arrays = np.vstack(
            roc_arrays
        )

        mean_tpr = roc_arrays.mean(axis=0)
        std_tpr = roc_arrays.std(axis=0)
        mean_tpr[-1] = 1

        axes[0].plot(
            fpr_grid,
            mean_tpr,
            color=BLUE,
            linewidth=2.3,
            label=(
                f"Mean ROC-AUC = "
                f"{fold_df['roc_auc'].mean():.3f}"
            )
        )

        axes[0].fill_between(
            fpr_grid,
            np.maximum(
                mean_tpr - std_tpr,
                0
            ),
            np.minimum(
                mean_tpr + std_tpr,
                1
            ),
            color=BLUE,
            alpha=0.16,
            linewidth=0,
            label="±1 SD across outer folds"
        )

        pr_arrays = []

        for precision_curve, recall_curve in result[
            "binary_pr_curves"
        ]:
            interpolated = np.interp(
                recall_grid,
                recall_curve[::-1],
                precision_curve[::-1]
            )

            pr_arrays.append(interpolated)

        pr_arrays = np.vstack(
            pr_arrays
        )

        mean_precision = pr_arrays.mean(axis=0)
        std_precision = pr_arrays.std(axis=0)

        axes[1].plot(
            recall_grid,
            mean_precision,
            color=ORANGE,
            linewidth=2.3,
            label=(
                f"Mean PR-AUC = "
                f"{fold_df['pr_auc'].mean():.3f}"
            )
        )

        axes[1].fill_between(
            recall_grid,
            np.maximum(
                mean_precision - std_precision,
                0
            ),
            np.minimum(
                mean_precision + std_precision,
                1
            ),
            color=ORANGE,
            alpha=0.16,
            linewidth=0,
            label="±1 SD across outer folds"
        )

        axes[1].axhline(
            y=y.mean(),
            linestyle="--",
            color=GRAY,
            linewidth=1,
            label=(
                f"Positive prevalence = "
                f"{y.mean():.2f}"
            )
        )

    else:
        for class_id in range(n_classes):
            class_color = CLASS_COLORS[
                class_id
            ]

            class_name = display_names[
                class_id
            ]

            roc_arrays = []

            for fpr, tpr in result[
                "multiclass_roc_curves"
            ][class_id]:
                interpolated = np.interp(
                    fpr_grid,
                    fpr,
                    tpr
                )

                interpolated[0] = 0
                roc_arrays.append(interpolated)

            roc_arrays = np.vstack(
                roc_arrays
            )

            mean_tpr = roc_arrays.mean(axis=0)
            std_tpr = roc_arrays.std(axis=0)
            mean_tpr[-1] = 1

            axes[0].plot(
                fpr_grid,
                mean_tpr,
                color=class_color,
                linewidth=2.1,
                label=class_name
            )

            axes[0].fill_between(
                fpr_grid,
                np.maximum(
                    mean_tpr - std_tpr,
                    0
                ),
                np.minimum(
                    mean_tpr + std_tpr,
                    1
                ),
                color=class_color,
                alpha=0.10,
                linewidth=0
            )

            pr_arrays = []

            for precision_curve, recall_curve in result[
                "multiclass_pr_curves"
            ][class_id]:
                interpolated = np.interp(
                    recall_grid,
                    recall_curve[::-1],
                    precision_curve[::-1]
                )

                pr_arrays.append(interpolated)

            pr_arrays = np.vstack(
                pr_arrays
            )

            mean_precision = pr_arrays.mean(axis=0)
            std_precision = pr_arrays.std(axis=0)

            axes[1].plot(
                recall_grid,
                mean_precision,
                color=class_color,
                linewidth=2.1,
                label=class_name
            )

            axes[1].fill_between(
                recall_grid,
                np.maximum(
                    mean_precision - std_precision,
                    0
                ),
                np.minimum(
                    mean_precision + std_precision,
                    1
                ),
                color=class_color,
                alpha=0.10,
                linewidth=0
            )

    axes[0].plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color=GRAY,
        linewidth=1
    )

    axes[0].set(
        xlabel="False positive rate",
        ylabel="True positive rate",
        title="a  ROC curve"
    )

    axes[1].set(
        xlabel="Recall",
        ylabel="Precision",
        title="b  Precision–recall curve"
    )

    for ax in axes:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)
        ax.legend()
        sns.despine(ax=ax)

    fig.suptitle(
        scenario_name,
        fontsize=14,
        fontweight="bold"
    )

    fig.tight_layout()

    save_figure(
        fig,
        task_dir,
        f"{label_column}_roc_pr_curves"
    )

    plt.show()
    plt.close(fig)

    true_all = predictions_df[
        "true_label_encoded"
    ].to_numpy()

    predicted_all = predictions_df[
        "predicted_label_encoded"
    ].to_numpy()

    cm_counts = confusion_matrix(
        true_all,
        predicted_all,
        labels=np.arange(n_classes)
    )

    cm_fraction = confusion_matrix(
        true_all,
        predicted_all,
        labels=np.arange(n_classes),
        normalize="true"
    )

    cm_annotations = np.empty_like(
        cm_fraction,
        dtype=object
    )

    for row in range(n_classes):
        for column in range(n_classes):
            cm_annotations[row, column] = (
                f"{cm_fraction[row, column] * 100:.1f}%"
                f"\n(n={cm_counts[row, column]})"
            )

    pd.DataFrame(
        cm_counts,
        index=[
            f"true_{name}"
            for name in display_names
        ],
        columns=[
            f"predicted_{name}"
            for name in display_names
        ]
    ).to_csv(
        task_dir /
        "confusion_matrix_counts.tsv",
        sep="\t"
    )

    figure_width = (
        5.4 if n_classes == 2 else 6.2
    )

    fig, ax = plt.subplots(
        figsize=(figure_width, 4.8)
    )

    sns.heatmap(
        cm_fraction * 100,
        annot=cm_annotations,
        fmt="",
        cmap=sns.light_palette(
            BLUE,
            as_cmap=True
        ),
        vmin=0,
        vmax=100,
        square=True,
        linewidths=1,
        linecolor="white",
        cbar_kws={
            "label": "Percentage within true class"
        },
        xticklabels=display_names,
        yticklabels=display_names,
        ax=ax
    )

    ax.set(
        xlabel="Predicted class",
        ylabel="True class",
        title=scenario_name
    )

    plt.setp(
        ax.get_xticklabels(),
        rotation=20,
        ha="right"
    )

    plt.setp(
        ax.get_yticklabels(),
        rotation=0
    )

    fig.tight_layout()

    save_figure(
        fig,
        task_dir,
        f"{label_column}_confusion_matrix"
    )

    plt.show()
    plt.close(fig)

    gain_summary = pd.DataFrame({
        "mean_gain": gain_df.mean(axis=1),
        "std_gain": gain_df.std(axis=1),
        "median_gain": gain_df.median(axis=1),
        "nonzero_fold_fraction": (
            gain_df > 0
        ).mean(axis=1),
    })

    gain_summary = gain_summary.sort_values(
        by=[
            "mean_gain",
            "nonzero_fold_fraction"
        ],
        ascending=[False, False],
        kind="mergesort"
    )

    gain_summary.index.name = "orthogroup"

    gain_summary.to_csv(
        task_dir /
        "xgbrf_gain_importance.tsv",
        sep="\t"
    )

    top_gain = (
        gain_summary
        .head(30)
        .sort_values(
            "mean_gain",
            ascending=True
        )
    )

    fig, ax = plt.subplots(
        figsize=(7.4, 11.5)
    )

    ax.barh(
        top_gain.index,
        top_gain["mean_gain"],
        color=BLUE,
        edgecolor="none",
        alpha=0.92,
        height=0.72
    )

    ax.set(
        xlabel="Mean gain importance across outer folds",
        ylabel="",
        title=(
            f"{scenario_name}: "
            "top 30 Orthogroups by XGBRF gain"
        )
    )

    ax.grid(
        axis="x",
        linestyle="--",
        linewidth=0.6,
        alpha=0.35
    )

    sns.despine(ax=ax)
    fig.tight_layout()

    save_figure(
        fig,
        task_dir,
        f"{label_column}_top30_gain_importance"
    )

    plt.show()
    plt.close(fig)

    if RUN_SHAP and shap_df is not None:
        shap_summary = pd.DataFrame({
            "mean_abs_shap": shap_df.mean(axis=1),
            "std_abs_shap": shap_df.std(axis=1),
            "median_abs_shap": shap_df.median(axis=1),
            "nonzero_fold_fraction": (
                shap_df > 0
            ).mean(axis=1),
        })

        shap_summary = shap_summary.sort_values(
            by=[
                "mean_abs_shap",
                "nonzero_fold_fraction"
            ],
            ascending=[False, False],
            kind="mergesort"
        )

        shap_summary.index.name = "orthogroup"

        shap_summary.to_csv(
            task_dir /
            "xgbrf_shap_importance.tsv",
            sep="\t"
        )

        top_shap = (
            shap_summary
            .head(30)
            .sort_values(
                "mean_abs_shap",
                ascending=True
            )
        )

        fig, ax = plt.subplots(
            figsize=(7.4, 11.5)
        )

        ax.barh(
            top_shap.index,
            top_shap["mean_abs_shap"],
            color=ORANGE,
            edgecolor="none",
            alpha=0.92,
            height=0.72
        )

        ax.set(
            xlabel="Mean absolute SHAP value across outer folds",
            ylabel="",
            title=(
                f"{scenario_name}: "
                "top 30 Orthogroups by SHAP importance"
            )
        )

        ax.grid(
            axis="x",
            linestyle="--",
            linewidth=0.6,
            alpha=0.35
        )

        sns.despine(ax=ax)
        fig.tight_layout()

        save_figure(
            fig,
            task_dir,
            f"{label_column}_top30_shap_importance"
        )

        plt.show()
        plt.close(fig)

    classification_report_df = pd.DataFrame(
        classification_report(
            true_all,
            predicted_all,
            labels=np.arange(n_classes),
            target_names=display_names,
            output_dict=True,
            zero_division=0
        )
    ).T

    classification_report_df.to_csv(
        task_dir /
        "classification_report.tsv",
        sep="\t"
    )

    print(
        f"{scenario_name} results saved to: "
        f"{task_dir}"
    )


In [ ]:
scenario_results = {}

for label_column in LABEL_COLUMNS:
    scenario_results[label_column] = (
        run_scenario(label_column)
    )

    save_and_plot_scenario(
        scenario_results[label_column]
    )

print(
    "\nAll scenarios completed. Output directory:"
    f"{OUTPUT_ROOT}"
)
